# v12 — Mass feature engineering + target encoding + 3-model stack

Combines the public 0.98 notebook (https://www.kaggle.com/code/rawashishsin/s6e4-highest-score-xgboost-cv-0-98109) 
with validated additions:
- XGB + LGB + CB instead of XGB only
- 3-seed averaging within each fold
- LR meta-learner stacking on OOF probabilities
- Engineered meta-features (per-class std, agreement, entropy)
- Multi-seed bounded class-weight tuning



In [ ]:
import gc
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from time import time
from itertools import combinations

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (balanced_accuracy_score,
                             classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_sample_weight
from scipy.optimize import minimize

from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
COMP_PATH    = Path('/kaggle/input/competitions/playground-series-s6e4/')
ORIG_PATH    = Path('/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset')
WORKING      = Path('/kaggle/working')

N_FOLDS         = 5
N_SEEDS         = 3
ORIG_ROW_WEIGHT = 0.5  
USE_GPU         = torch.cuda.is_available()

print(f'GPU: {USE_GPU}')

## Load data and prepare target

In [ ]:
train = pd.read_csv(COMP_PATH / 'train.csv')
test  = pd.read_csv(COMP_PATH / 'test.csv')
orig  = pd.read_csv(next(ORIG_PATH.glob('*.csv')))
for c in ['Id', 'ID']:
    if c in orig.columns:
        orig = orig.drop(columns=[c])
if 'id' in orig.columns:
    orig = orig.drop(columns=['id'])

TARGET = 'Irrigation_Need'
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
for df in [train, orig]:
    df[TARGET] = df[TARGET].map(target_map)

test_ids = test['id'].copy()

cat_cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
            'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
num_cols = ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
            'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
            'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']

# Top features identified by null importance + the rule reverse-engineering
top_cat_cols = ['Crop_Growth_Stage', 'Mulching_Used', 'Crop_Type']
top_num_cols = ['Soil_Moisture', 'Temperature_C', 'Wind_Speed_kmh', 'Rainfall_mm']
top_cols     = ['Soil_Moisture', 'Crop_Growth_Stage', 'Temperature_C',
                'Mulching_Used', 'Wind_Speed_kmh', 'Rainfall_mm']

print(f'train: {train.shape}, test: {test.shape}, orig: {orig.shape}')

## Feature engineering


In [ ]:
train_fe = train.copy()
test_fe  = test.copy()
orig_fe  = orig.copy()

def add_threshold_and_domain_features(df):
    df['soil_lt_25']  = (df['Soil_Moisture']  < 25).astype(int)
    df['wind_gt_10']  = (df['Wind_Speed_kmh'] > 10).astype(int)
    df['temp_gt_30']  = (df['Temperature_C']  > 30).astype(int)
    df['rain_lt_300'] = (df['Rainfall_mm']    < 300).astype(int)
    df['moist_rain']    = df['Soil_Moisture'] / (df['Rainfall_mm'] + 1)
    df['moist_temp']    = df['Soil_Moisture'] / (df['Temperature_C'] + 1)
    df['moist_wind']    = df['Soil_Moisture'] / (df['Wind_Speed_kmh'] + 1)
    df['ET_proxy']      = (df['Temperature_C'] * df['Wind_Speed_kmh'] * df['Sunlight_Hours']) / (df['Humidity'] + 1)
    df['heat_stress']   = df['Temperature_C'] * df['Sunlight_Hours']
    df['drying_force']  = df['Wind_Speed_kmh'] * df['Temperature_C'] / (df['Humidity'] + 1)
    df['water_supply']  = df['Rainfall_mm'] + df['Previous_Irrigation_mm']
    df['water_deficit'] = df['Soil_Moisture'] - df['water_supply'] * 0.1
    df['soil_quality']  = df['Organic_Carbon'] / (df['Electrical_Conductivity'] + 0.1)
    df['moist_x_temp']  = df['Soil_Moisture'] * df['Temperature_C']
    df['wind_x_temp']   = df['Wind_Speed_kmh'] * df['Temperature_C']
    return df

def add_formula_score(df):
    df['high_score'] = ((df['Soil_Moisture'] < 25) * 2 +
                        (df['Rainfall_mm'] < 300) * 2 +
                        (df['Temperature_C'] > 30) * 1 +
                        (df['Wind_Speed_kmh'] > 10) * 1)
    df['low_score']  = ((df['Crop_Growth_Stage'].isin(['Harvest', 'Sowing'])) * 2 +
                        (df['Mulching_Used'] == 'Yes') * 1)
    df['formula_score'] = df['high_score'] - df['low_score']
    df['formula_pred']  = 0
    df.loc[(df['formula_score'] > 0) & (df['formula_score'] <= 3), 'formula_pred'] = 1
    df.loc[df['formula_score'] > 3, 'formula_pred'] = 2
    return df

for df in [train_fe, test_fe, orig_fe]:
    add_threshold_and_domain_features(df)
    add_formula_score(df)

In [ ]:
# N-gram categoricals: bigrams of all top_cat pairs, trigrams of all triples
BIGRAM_COLS, TRIGRAM_COLS = [], []
frames = [train_fe, test_fe, orig_fe]

for c1, c2 in combinations(top_cat_cols, 2):
    name = f'BG_{c1}_{c2}'
    for df in frames:
        df[name] = df[c1].astype(str) + '_' + df[c2].astype(str)
    BIGRAM_COLS.append(name)

for c1, c2, c3 in combinations(top_cat_cols, 3):
    name = f'TG_{c1}_{c2}_{c3}'
    for df in frames:
        df[name] = df[c1].astype(str) + '_' + df[c2].astype(str) + '_' + df[c3].astype(str)
    TRIGRAM_COLS.append(name)

NGRAM_COLS = BIGRAM_COLS + TRIGRAM_COLS

# Factorize n-grams using combined train+test+orig vocab so codes align across splits
for col in NGRAM_COLS:
    combined = pd.concat([train_fe[col], test_fe[col], orig_fe[col]], axis=0).astype(str)
    labels, _ = pd.factorize(combined)
    n_train, n_test = len(train_fe), len(test_fe)
    train_fe[col] = labels[:n_train]
    test_fe[col]  = labels[n_train:n_train + n_test]
    orig_fe[col]  = labels[n_train + n_test:]

print(f'n-gram features: {len(NGRAM_COLS)}')

In [ ]:
# Binned numerical × categorical interactions
BIN_CAT_INT_COLS = []
for num_col in top_num_cols:
    bin_col = f'{num_col}_bin'
    train_fe[bin_col], bins = pd.qcut(train_fe[num_col], q=5, labels=False,
                                       retbins=True, duplicates='drop')
    for df in [test_fe, orig_fe]:
        df[bin_col] = pd.cut(df[num_col], bins=bins, labels=False,
                             include_lowest=True).fillna(0).astype(int)
    for cat_col in top_cat_cols:
        name = f'{num_col}_bin_{cat_col}_int'
        for df in frames:
            df[name] = df[bin_col].astype(str) + '_' + df[cat_col].astype(str)
        BIN_CAT_INT_COLS.append(name)

for df in frames:
    df.drop(columns=[f'{c}_bin' for c in top_num_cols], inplace=True)

# Factorize using train vocab (test/orig get mapped via the same dict)
for col in BIN_CAT_INT_COLS:
    codes, uniques = pd.factorize(train_fe[col])
    train_fe[col] = codes.astype(int)
    mapping = {v: i for i, v in enumerate(uniques)}
    test_fe[col] = test_fe[col].map(mapping).fillna(-1).astype(int)
    orig_fe[col] = orig_fe[col].map(mapping).fillna(-1).astype(int)

print(f'binned-cat interaction features: {len(BIN_CAT_INT_COLS)}')

In [ ]:
# Group aggregates: per-cat mean of each numeric, plus diff and ratio
NUM_CAT_AGG_COLS = []
for cat_col in top_cat_cols:
    for num_col in top_num_cols:
        group_means = train_fe.groupby(cat_col)[num_col].mean()
        for df in frames:
            avg   = f'{num_col}_avg_by_{cat_col}'
            diff  = f'{num_col}_diff_{cat_col}'
            ratio = f'{num_col}_ratio_{cat_col}'
            df[avg]   = df[cat_col].map(group_means).astype(float)
            df[diff]  = (df[num_col] - df[avg]).astype(float)
            df[ratio] = (df[num_col] / (df[avg] + 1e-6)).astype(float)
            NUM_CAT_AGG_COLS.extend([avg, diff, ratio])
NUM_CAT_AGG_COLS = list(set(NUM_CAT_AGG_COLS))
print(f'group aggregate features: {len(NUM_CAT_AGG_COLS)}')

In [ ]:
# Round, digit, decimal, and bin features on top numerics
round_config  = {'Soil_Moisture': [0, -1], 'Temperature_C': [-1],
                 'Rainfall_mm': [0, -1, -2, -3], 'Wind_Speed_kmh': [0, -1]}
digit_config  = {'Soil_Moisture': [-1, 0, 1, 2], 'Temperature_C': [-1, 0, 1, 2],
                 'Rainfall_mm': [-3, -2, -1, 0, 1, 2], 'Wind_Speed_kmh': [-1, 0, 1, 2]}

ROUND, DIGITS, DECIMALS, BINS = [], [], [], []

for col, rs in round_config.items():
    for r in rs:
        feat = f'{col}_r{r}'
        for df in frames:
            df[feat] = df[col].round(r)
        ROUND.append(feat)

for col, ks in digit_config.items():
    for k in ks:
        feat = f'{col}_d{k}'
        for df in frames:
            df[feat] = ((df[col] * 10**k) % 10).astype(int)
        DIGITS.append(feat)

for col in top_num_cols:
    feat = f'{col}_decimal'
    for df in frames:
        df[feat] = (df[col] % 1).round(2)
    DECIMALS.append(feat)

for col in top_num_cols:
    feat = f'{col}_qbin'
    train_fe[feat], edges = pd.qcut(train_fe[col], q=10, labels=False,
                                     duplicates='drop', retbins=True)
    for df in [test_fe, orig_fe]:
        df[col] = df[col].clip(edges[0], edges[-1])
        df[feat] = pd.cut(df[col], bins=edges, labels=False, include_lowest=True).astype(int)
    BINS.append(feat)

print(f'round={len(ROUND)}, digits={len(DIGITS)}, decimals={len(DECIMALS)}, bins={len(BINS)}')

In [ ]:
# Pairwise factorized features of top categoricals + numerics
PAIRS = []
n_train, n_test, n_orig = len(train_fe), len(test_fe), len(orig_fe)
n_total = n_train + n_test + n_orig

for c1, c2 in combinations(top_cols, 2):
    name = f'{c1}__{c2}'
    combined = pd.concat([
        train_fe[c1].astype(str) + '_' + train_fe[c2].astype(str),
        test_fe[c1].astype(str)  + '_' + test_fe[c2].astype(str),
        orig_fe[c1].astype(str)  + '_' + orig_fe[c2].astype(str)
    ], ignore_index=True)
    codes, _ = pd.factorize(combined)
    if len(np.unique(codes)) > n_total // 2 or len(np.unique(codes)) <= 1:
        continue
    train_fe[name] = codes[:n_train]
    test_fe[name]  = codes[n_train:n_train + n_test]
    orig_fe[name]  = codes[n_train + n_test:]
    PAIRS.append(name)

print(f'pair features: {len(PAIRS)}')
gc.collect()

In [ ]:
# Convert original cat columns to category dtype for XGB enable_categorical
for df in frames:
    for col in cat_cols:
        df[col] = df[col].astype('category')

X_train = train_fe.drop(columns=[TARGET, 'id'])
y_train = train_fe[TARGET].values
X_test  = test_fe.drop(columns=['id'])
X_orig  = orig_fe.drop(columns=[TARGET])
y_orig  = orig_fe[TARGET].values

print(f'X_train: {X_train.shape}  X_test: {X_test.shape}  X_orig: {X_orig.shape}')
print(f'total features: {X_train.shape[1]}')

## Training loop with in-fold target encoding

For each fold:
1. Split train into fit/val.
2. Concat fit set with original.
3. Multiclass target-encode every feature column inside the fold (3 new columns per feature: P(Low|x), P(Medium|x), P(High|x)).
4. Drop high-cardinality engineered columns that target encoding has already condensed (PAIRS, NUM_CAT_AGG_COLS, BIN_CAT_INT_COLS).
5. Train XGB, LGB, CB across 3 seeds. Average within fold.
6. Save OOF + test probabilities. Checkpoint after each fold.

The 0.98 notebook used XGB only, I add LGB and CB for ensemble diversity

In [ ]:
# These get dropped after target encoding because they're high-cardinality and TE already absorbed their signal
DROP_AFTER_TE = PAIRS + NUM_CAT_AGG_COLS + BIN_CAT_INT_COLS
TE_FEATURES   = X_train.columns.tolist()

params_xgb = {
    'objective': 'multi:softprob', 'num_class': 3,
    'n_estimators': 2600, 'learning_rate': 0.05,
    'max_depth': 3, 'subsample': 0.9, 'colsample_bytree': 0.8,
    'max_bin': 1100, 'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'device': 'cuda' if USE_GPU else 'cpu',
    'enable_categorical': True, 'n_jobs': -1, 'verbosity': 0,
}

params_lgb = {
    'objective': 'multiclass', 'num_class': 3,
    'n_estimators': 2600, 'learning_rate': 0.05,
    'num_leaves': 15,            # depth ~3-4 equivalent
    'min_child_samples': 30,
    'subsample': 0.9, 'subsample_freq': 1, 'colsample_bytree': 0.8,
    'reg_alpha': 0.3, 'reg_lambda': 0.3,
    'verbosity': -1, 'n_jobs': -1,
}
if USE_GPU:
    params_lgb['device'] = 'gpu'

params_cb = {
    'iterations': 2600, 'learning_rate': 0.05,
    'depth': 4, 'l2_leaf_reg': 3.0,
    'subsample': 0.9, 'bootstrap_type': 'Bernoulli',
    'loss_function': 'MultiClass', 'classes_count': 3,
    'verbose': 0,
}
if USE_GPU:
    params_cb['task_type'] = 'GPU'

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

MODELS = ['xgb', 'lgb', 'cb']
oof_proba  = {m: np.zeros((len(X_train), 3)) for m in MODELS}
test_proba = {m: np.zeros((len(X_test),  3)) for m in MODELS}
fold_bas   = {m: [] for m in MODELS}

OOF_FILE  = WORKING / 'v12_oof.npz'
TEST_FILE = WORKING / 'v12_test.npz'

t0 = time()
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
    print(f'\n=== fold {fold+1}/{N_FOLDS} ===')
    X_tr_real = X_train.iloc[tr_idx]
    y_tr_real = y_train[tr_idx]
    X_val     = X_train.iloc[va_idx]
    y_val     = y_train[va_idx]

    # Combine real train fold with original-dataset rows for fitting
    X_tr_combined = pd.concat([X_tr_real, X_orig], axis=0).reset_index(drop=True)
    y_tr_combined = np.concatenate([y_tr_real, y_orig])

    # In-fold multiclass target encoding (adds 3 cols per feature)
    encoder = TargetEncoder(target_type='multiclass', smooth='auto',
                            cv=5, random_state=RANDOM_STATE)
    X_tr_te   = encoder.fit_transform(X_tr_combined[TE_FEATURES], y_tr_combined)
    X_val_te  = encoder.transform(X_val[TE_FEATURES])
    X_test_te = encoder.transform(X_test[TE_FEATURES])

    te_cols = [f'TE_{c}_class{k}' for c in TE_FEATURES for k in range(3)]
    X_tr_combined[te_cols] = X_tr_te
    X_val_local  = X_val.copy()
    X_val_local[te_cols] = X_val_te
    X_test_local = X_test.copy()
    X_test_local[te_cols] = X_test_te

    # Drop the high-card columns now that TE has condensed them
    X_tr_final  = X_tr_combined.drop(columns=DROP_AFTER_TE)
    X_val_final = X_val_local.drop(columns=DROP_AFTER_TE)
    X_te_final  = X_test_local.drop(columns=DROP_AFTER_TE)

    # Sample weights: balanced by class, then orig rows downweighted
    sw = compute_sample_weight('balanced', y_tr_combined)
    sw[len(tr_idx):] *= ORIG_ROW_WEIGHT

    # XGB seed averaging
    seed_val_xgb  = np.zeros((len(X_val), 3))
    seed_test_xgb = np.zeros((len(X_test), 3))
    for seed in range(N_SEEDS):
        m = XGBClassifier(**{**params_xgb, 'random_state': RANDOM_STATE + seed})
        m.fit(X_tr_final, y_tr_combined,
              eval_set=[(X_val_final, y_val)], sample_weight=sw, verbose=False)
        seed_val_xgb  += m.predict_proba(X_val_final) / N_SEEDS
        seed_test_xgb += m.predict_proba(X_te_final)  / N_SEEDS
    oof_proba['xgb'][va_idx] = seed_val_xgb
    test_proba['xgb']      += seed_test_xgb / N_FOLDS
    ba_xgb = balanced_accuracy_score(y_val, seed_val_xgb.argmax(axis=1))
    fold_bas['xgb'].append(ba_xgb)
    print(f'  xgb: BA = {ba_xgb:.5f}  [{(time()-t0)/60:.1f} min]')

    # LGB seed averaging — convert categoricals to int codes since LGB doesn't share XGB's categorical handling
    X_tr_lgb  = X_tr_final.copy()
    X_val_lgb = X_val_final.copy()
    X_te_lgb  = X_te_final.copy()
    for c in cat_cols:
        if c in X_tr_lgb.columns:
            X_tr_lgb[c]  = X_tr_lgb[c].cat.codes.astype(int)
            X_val_lgb[c] = X_val_lgb[c].cat.codes.astype(int)
            X_te_lgb[c]  = X_te_lgb[c].cat.codes.astype(int)

    seed_val_lgb  = np.zeros((len(X_val), 3))
    seed_test_lgb = np.zeros((len(X_test), 3))
    for seed in range(N_SEEDS):
        m = lgb.LGBMClassifier(**{**params_lgb, 'random_state': RANDOM_STATE + seed})
        m.fit(X_tr_lgb, y_tr_combined, sample_weight=sw,
              eval_set=[(X_val_lgb, y_val)],
              callbacks=[lgb.early_stopping(150, verbose=False)])
        seed_val_lgb  += m.predict_proba(X_val_lgb) / N_SEEDS
        seed_test_lgb += m.predict_proba(X_te_lgb)  / N_SEEDS
    oof_proba['lgb'][va_idx] = seed_val_lgb
    test_proba['lgb']      += seed_test_lgb / N_FOLDS
    ba_lgb = balanced_accuracy_score(y_val, seed_val_lgb.argmax(axis=1))
    fold_bas['lgb'].append(ba_lgb)
    print(f'  lgb: BA = {ba_lgb:.5f}  [{(time()-t0)/60:.1f} min]')

    # CatBoost seed averaging — pass cat_features by name
    cat_features_cb = [c for c in cat_cols if c in X_tr_final.columns]
    X_tr_cb  = X_tr_final.copy()
    X_val_cb = X_val_final.copy()
    X_te_cb  = X_te_final.copy()
    for c in cat_features_cb:
        X_tr_cb[c]  = X_tr_cb[c].astype(str)
        X_val_cb[c] = X_val_cb[c].astype(str)
        X_te_cb[c]  = X_te_cb[c].astype(str)

    seed_val_cb  = np.zeros((len(X_val), 3))
    seed_test_cb = np.zeros((len(X_test), 3))
    for seed in range(N_SEEDS):
        m = CatBoostClassifier(**{**params_cb, 'random_seed': RANDOM_STATE + seed,
                                   'early_stopping_rounds': 150})
        m.fit(X_tr_cb, y_tr_combined, sample_weight=sw,
              eval_set=(X_val_cb, y_val), cat_features=cat_features_cb)
        seed_val_cb  += m.predict_proba(X_val_cb) / N_SEEDS
        seed_test_cb += m.predict_proba(X_te_cb)  / N_SEEDS
    oof_proba['cb'][va_idx] = seed_val_cb
    test_proba['cb']      += seed_test_cb / N_FOLDS
    ba_cb = balanced_accuracy_score(y_val, seed_val_cb.argmax(axis=1))
    fold_bas['cb'].append(ba_cb)
    print(f'  cb:  BA = {ba_cb:.5f}  [{(time()-t0)/60:.1f} min]')

    # Checkpoint
    np.savez(OOF_FILE,  **oof_proba)
    np.savez(TEST_FILE, **test_proba)

    del X_tr_final, X_val_final, X_te_final, X_tr_combined, X_val_local, X_test_local
    del X_tr_lgb, X_val_lgb, X_te_lgb, X_tr_cb, X_val_cb, X_te_cb
    gc.collect()

print(f'\ntotal training time: {(time()-t0)/60:.1f} min')
for m in MODELS:
    bas = np.array(fold_bas[m])
    ba_oof = balanced_accuracy_score(y_train, oof_proba[m].argmax(axis=1))
    print(f'{m}: per-fold BA = {bas.mean():.5f} ± {bas.std():.5f}  |  full OOF = {ba_oof:.5f}')

## Stacking: LR meta-learner with engineered meta-features

In [ ]:
def compute_meta_features(model_probas):
    """Per-row diagnostics across base-model probabilities."""
    stacked = np.stack([model_probas[m] for m in model_probas], axis=0)
    n = stacked.shape[1]
    mean_proba = stacked.mean(axis=0)
    std_proba  = stacked.std(axis=0)
    feats = pd.DataFrame(index=range(n))
    for c, name in enumerate(['Low', 'Medium', 'High']):
        feats[f'mean_{name}'] = mean_proba[:, c]
        feats[f'std_{name}']  = std_proba[:, c]
        feats[f'var_{name}']  = std_proba[:, c] ** 2
    sorted_mean = np.sort(mean_proba, axis=1)
    feats['margin']    = sorted_mean[:, -1] - sorted_mean[:, -2]
    feats['max_prob']  = sorted_mean[:, -1]
    feats['entropy']   = -np.sum(mean_proba * np.log(mean_proba + 1e-9), axis=1)
    argmaxes = stacked.argmax(axis=2)
    majority = mean_proba.argmax(axis=1)
    feats['agreement'] = (argmaxes == majority[None, :]).mean(axis=0)
    return feats.astype(np.float32)

meta_oof  = compute_meta_features(oof_proba)
meta_test = compute_meta_features(test_proba)

X_meta_oof  = np.hstack([np.hstack([oof_proba[m]  for m in MODELS]), meta_oof.values])
X_meta_test = np.hstack([np.hstack([test_proba[m] for m in MODELS]), meta_test.values])

skf_meta = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE+999)
stack_oof  = np.zeros((len(X_meta_oof), 3))
stack_test = np.zeros((len(X_meta_test), 3))
for tr_idx, va_idx in skf_meta.split(X_meta_oof, y_train):
    meta = LogisticRegression(C=1.0, max_iter=500, n_jobs=-1)
    sw_meta = compute_sample_weight('balanced', y_train[tr_idx])
    meta.fit(X_meta_oof[tr_idx], y_train[tr_idx], sample_weight=sw_meta)
    stack_oof[va_idx] = meta.predict_proba(X_meta_oof[va_idx])
    stack_test += meta.predict_proba(X_meta_test) / N_FOLDS

stack_ba = balanced_accuracy_score(y_train, stack_oof.argmax(axis=1))
ens_oof  = np.mean([oof_proba[m]  for m in MODELS], axis=0)
ens_test = np.mean([test_proba[m] for m in MODELS], axis=0)
ens_ba   = balanced_accuracy_score(y_train, ens_oof.argmax(axis=1))

print(f'equal-weight ensemble OOF BA: {ens_ba:.5f}')
print(f'stacked + meta-features  OOF BA: {stack_ba:.5f}')

## Multi-seed bounded class-weight tuning

In [ ]:
def neg_ba(weights, proba, y_true):
    return -balanced_accuracy_score(y_true, (proba * weights).argmax(axis=1))

BOUNDS = [(0.7, 1.5)] * 3
TUNING_SEEDS = [0, 1, 2, 3, 4]

def multi_seed_tune(oof_proba_in, y_true, label):
    raw_ba = balanced_accuracy_score(y_true, oof_proba_in.argmax(axis=1))
    print(f'{label}: raw OOF BA = {raw_ba:.5f}')
    all_w = []
    for seed in TUNING_SEEDS:
        rng = np.random.default_rng(seed)
        idx = np.arange(len(y_true)); rng.shuffle(idx)
        half = len(idx) // 2
        f1, f2 = idx[:half], idx[half:]
        seed_w = []
        for tune_idx, _ in [(f1, f2), (f2, f1)]:
            best = (None, -np.inf)
            for x0 in [[1,1,1], [1.2,0.9,1], [1.4,0.85,1.05]]:
                res = minimize(neg_ba, x0=x0,
                               args=(oof_proba_in[tune_idx], y_true[tune_idx]),
                               method='Nelder-Mead', bounds=BOUNDS,
                               options={'xatol': 1e-4, 'maxiter': 300})
                if -res.fun > best[1]:
                    best = (res.x, -res.fun)
            seed_w.append(best[0])
        all_w.append(np.mean(seed_w, axis=0))
    final = np.mean(all_w, axis=0)
    final_ba = balanced_accuracy_score(y_true, (oof_proba_in * final).argmax(axis=1))
    print(f'  averaged weights: {final.round(4)}  tuned OOF BA: {final_ba:.5f}  (gain: {final_ba - raw_ba:+.5f})')
    return final

stack_w = multi_seed_tune(stack_oof, y_train, 'stacked')

## Final reports and submissions

In [ ]:
stack_tuned_pred = (stack_oof * stack_w).argmax(axis=1)
print(f'final stack tuned OOF BA: {balanced_accuracy_score(y_train, stack_tuned_pred):.5f}')
print()
print(classification_report(y_train, stack_tuned_pred,
                            target_names=['Low', 'Medium', 'High'], digits=4))
print(confusion_matrix(y_train, stack_tuned_pred))

reverse_map = {0: 'Low', 1: 'Medium', 2: 'High'}

def make_sub(proba, name):
    pred = proba.argmax(axis=1)
    sub = pd.DataFrame({'id': test_ids, 'Irrigation_Need': [reverse_map[p] for p in pred]})
    sub.to_csv(WORKING / f'submission_{name}.csv', index=False)
    print(f'submission_{name}.csv: {sub["Irrigation_Need"].value_counts().to_dict()}')

make_sub(stack_test,           'v12_stack_raw')
make_sub(stack_test * stack_w, 'v12_stack_tuned')
make_sub(ens_test,             'v12_ens_raw')

np.savez(WORKING / 'v12_stack.npz',
         stack_oof=stack_oof, stack_test=stack_test,
         ens_oof=ens_oof, ens_test=ens_test,
         stack_weights=stack_w)